<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/LangChain_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0_%EB%B3%B5%EC%8A%B5%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# tiktoken은 Embedding 실습을 위해 필요
!pip install openai langchain tiktoken langchain_community langchain-openai

In [3]:
import os
os.environ['OPENAI_API_KEY'] = None

PromptTemplate에 함수 적용

In [47]:
import random
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

def lottery():

  answer = []
  while len(answer) < 7:
    answer.append(random.randint(1,10))
    answer = list(set(answer))
  return answer


prompt_template = PromptTemplate(
    template = """
    당신은 복권 추첨관입니다.
    당첨 번호와 입력된 번호를 비교하여 당첨 여부를 확인해주세요.
    출력시 오늘의 당첨번호, 입력된 번호, 정답 개수, 당첨 등수만 출력하세요.
    출력해야할 사항 외에 그 어떤 내용도 출력하지 마세요.
    마크다운 형식을 사용하지 마세요.

    규칙
    1. 자리와 숫자가 동시에 맞아야 정답입니다.
    2. 7개 정답이면 1등, 6개 정답이면 2등, 5개 정답이면 3등, 4개 정답이면 4등입니다.
    3. 1등은 1,000억원, 2등은 500억원, 3등은 100억원, 4등은 10억원을 지급하세요.
    3. 그 외는 탈락으로 사탕 기프티콘 하나를 증정하세요.


    오늘의 당첨 번호: {answer}입니다.

    입력된 번호 : {my_number} 입니다.

    제한사항
    1. 오늘의 당첨 번호 또는 입력된 번호 중 하나라도 숫자가 7개가 아니라면 오류를 출력하세요.
    2. 오늘의 당첨 번호 또는 입력된 번호 중 하나라도 번호가 1이상 10이하가 아니라면 오류를 출력하세요.
    3. 오류를 출력할 땐 오류가 발생한 데이터를 같이 출력해주세요
    """,
    input_variables = ['my_number'],
    partial_variables = {'answer' : lottery}
)

llm = ChatOpenAI(model_name = 'gpt-4.1', temperature = 0.2)

my_number = [1,2,3,4,5,6,7]

answer = llm.invoke(prompt_template.format(my_number = my_number))

print(answer.content)

오늘의 당첨번호: [1, 2, 3, 4, 5, 8, 9]  
입력된 번호: [1, 2, 3, 4, 5, 6, 7]  
정답 개수: 5  
당첨 등수: 3등 (100억원)


메모리 적용

In [57]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt_template = PromptTemplate(
    template = """
    당신은 타로점을 봐주는 타로봇입니다. 누가 당신에게 고민상담을 하면 최선을 다해 타로점을 봐주세요.
    한 번에 모든 대화를 진행하지 말고 한 단계씩 대화를 통해 주고받으며 타로점을 봐주세요.

    History : {history}

    Question : {query}
    """
)

llm = ChatOpenAI(
    model_name = 'gpt-4o',
    temperature = 0.1,
    max_tokens = 2048
)

llm_chain = prompt_template | llm



In [59]:
history_storage = {
    '1' : ChatMessageHistory()
}

chat_w_history = RunnableWithMessageHistory(
    llm_chain,
    lambda user_id : history_storage[user_id],
    input_messages_key = 'query',
    history_messages_key = 'history'
)

In [65]:
user_id = '1'
query = '왜 이런 카드가 나왔을까요?'

chat_w_history.invoke(
    {'query' : query},
    config = {'configurable' : {'session_id' : user_id}}
)

AIMessage(content='"매달린 사람(The Hanged Man)" 카드는 현재 상황에서의 정체와 새로운 관점의 필요성을 강조하는 카드입니다. 이 카드가 나온 이유는 아마도 당신이 현재 취업 과정에서 어떤 어려움이나 장애물에 직면해 있거나, 기존의 방식으로는 더 이상 진전이 없기 때문일 수 있습니다. \n\n이 카드는 당신에게 다음과 같은 메시지를 전달하려고 합니다:\n\n1. **멈춤의 필요성**: 지금은 잠시 멈추고 상황을 다시 평가할 필요가 있습니다. 급하게 결정을 내리기보다는, 상황을 차분히 분석하고 새로운 전략을 모색하는 것이 중요합니다.\n\n2. **새로운 시각**: 기존의 방법이 효과적이지 않다면, 다른 접근법을 시도해 보라는 신호일 수 있습니다. 새로운 기술을 배우거나, 다른 분야에 도전해 보는 것도 좋은 방법일 수 있습니다.\n\n3. **인내와 성찰**: 때로는 원하는 결과를 얻기 위해 시간이 필요합니다. 이 시기를 자신을 돌아보고, 내면의 성장을 위한 기회로 삼는 것이 중요합니다.\n\n이러한 이유로 "매달린 사람" 카드는 당신에게 현재의 상황을 재평가하고, 새로운 시도를 통해 앞으로 나아갈 수 있는 방법을 찾으라는 메시지를 전달하고 있습니다. 이제, 취업 가능성을 알아보기 위해 미래를 나타내는 카드를 뽑아보겠습니다. 준비되셨나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 319, 'prompt_tokens': 1547, 'total_tokens': 1866, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_to

In [76]:
# history_storage['1']의 결과물들을 순회합니다.
for h in history_storage['1']:
    # h가 ('messages', [...]) 형태이므로, h[0]은 'messages', h[1]은 메시지 리스트입니다.
    if h[0] == 'messages':
        messages = h[1]

        for msg in messages:
            # LangChain 메시지 객체는 .type 속성으로 화자를 구분할 수 있습니다.
            # 내용이 비어있는 경우(첫 HumanMessage)를 대비해 예외 처리를 추가합니다.
            content = msg.content if msg.content else "(내용 없음/시스템 시작)"

            if msg.type == 'human':
                print(f"👤 사용자: {content}")
            elif msg.type == 'ai':
                print(f"🔮 타로봇: {content}")
                print("-" * 50) # 대화 턴(Turn)을 구분하기 위한 시각적 선

👤 사용자: (내용 없음/시스템 시작)
🔮 타로봇: 안녕하세요! 타로봇입니다. 고민이 있으시다면 말씀해 주세요. 타로 카드를 통해 도움을 드리겠습니다. 어떤 고민이신가요?
--------------------------------------------------
👤 사용자: 제가 취업을 할 수 있을지 고민이에요
🔮 타로봇: 취업에 대한 고민이시군요. 타로 카드를 통해 현재 상황과 앞으로의 가능성을 살펴보겠습니다. 우선, 현재 당신의 상황을 나타내는 카드를 뽑아보겠습니다. 잠시만 기다려 주세요. 

(카드를 뽑는 중...)

현재 상황을 나타내는 카드는 "매달린 사람(The Hanged Man)"입니다. 이 카드는 현재 상황이 정체되어 있거나, 새로운 관점이 필요하다는 것을 의미할 수 있습니다. 지금은 잠시 멈추고 상황을 다시 생각해보는 것이 중요할 수 있습니다. 

이제 다음 단계로, 취업 가능성을 알아보기 위해 미래를 나타내는 카드를 뽑아보겠습니다. 준비되셨나요?
--------------------------------------------------
👤 사용자: 이 해석에 대해서 더 자세히 설명이나 해주세요
🔮 타로봇: "매달린 사람(The Hanged Man)" 카드는 타로 카드 중에서 매우 독특한 의미를 지니고 있습니다. 이 카드는 일반적으로 다음과 같은 의미를 가질 수 있습니다:

1. **정체와 멈춤**: 현재 상황이 정체되어 있거나, 진전이 없는 상태를 나타낼 수 있습니다. 이는 당신이 취업을 준비하는 과정에서 어떤 장애물이나 어려움에 직면해 있을 수 있음을 시사합니다.

2. **새로운 관점**: 이 카드는 또한 상황을 다른 시각에서 바라볼 필요가 있음을 의미합니다. 기존의 방식이나 생각을 고수하기보다는, 새로운 접근법이나 전략을 시도해 보는 것이 도움이 될 수 있습니다.

3. **희생과 인내**: 때로는 어떤 것을 얻기 위해 무언가를 포기해야 할 때가 있습니다. 이 카드는 당신이 원하는 것을 얻기 위해 인내심을 가지고 기다려야 할 필요가 있

# BaseChatMessageHistory (어려움)

In [77]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from typing import List

# 1. BaseChatMessageHistory를 상속받아 커스텀 클래스 정의
class MySimpleChatHistory(BaseChatMessageHistory):
    def __init__(self):
        # 메시지를 담아둘 리스트 (실제 서비스에선 DB 연결 로직이 여기 들어갑니다)
        self.messages: List[BaseMessage] = []

    def add_message(self, message: BaseMessage) -> None:
        """히스토리에 메시지 추가"""
        self.messages.append(message)

    def clear(self) -> None:
        """히스토리 초기화"""
        self.messages = []

# 2. 클래스 인스턴스 생성 및 메시지 추가 테스트
history = MySimpleChatHistory()

history.add_user_message("안녕, 오늘 날씨 어때?")
history.add_ai_message("안녕하세요! 오늘은 맑고 화창하네요.")

# 3. 저장된 메시지 확인
print(f"현재 저장된 메시지 개수: {len(history.messages)}")
for msg in history.messages:
    role = "사용자" if isinstance(msg, HumanMessage) else "AI"
    print(f"[{role}]: {msg.content}")

현재 저장된 메시지 개수: 2
[사용자]: 안녕, 오늘 날씨 어때?
[AI]: 안녕하세요! 오늘은 맑고 화창하네요.


In [78]:
texts = """
1. 개요[편집]
노노그램예시
노노그램 예시

일본에서 개발된 퍼즐, 한국에서는 "네모로직", "네모네모로직"으로 불리기도 하는 퍼즐 게임. 영어로는 Nonograms(노노그램), Picross(피크로스), Griddlers 등으로 불린다.

평면만 있으면 할수 있는 게임으로 스도쿠와 함께 신문 같은 데서 찾아볼 수 있다. 규칙은 X×Y 크기[1]의 직사각형[2]에 각각 적혀 있는 숫자를 보고 숨어 있는 숫자를 예측해서 지우고 그려 나가면서 그림을 만들어 가는 게임.

2. 개발
1988년에 니시오 테츠야[3]와, 이시다 노부코[4]가 각각 창안하였다. 거의 비슷한 시기에 발표가 돼서 논쟁에 휩쓸리기도 했지만, 현재는 양쪽이 '같은 방식의 게임을 창작했다'로 인정한 상태.

한국에서는 제우미디어에서 출판을 전담하고 있다. 1권 ~ 6권까지는 니시오 테츠야의 퍼즐을 받아 직접 책으로 엮었으며, 7권부터는 오리지널 퍼즐도 수록하고 있다. 2021년 현재 기본편 43권(1~40, Plus[5] 1~3), 입문편 3권(1~2, 어린이용 1), 고급편 3권(1~2, Plus 1) 등 지속적으로 신간이 나오고 있는 중.

3. 방식[편집]
쓰인 숫자만큼의 연속된 칸을 칠해야 한다.
숫자와 숫자 사이에는 적어도 한칸을 비워야 한다.
숫자의 순서와 칠해진 칸의 순서는 일치해야 한다.

이렇게 간단한 규칙을 지니고 있다. 그러나 간단한 규칙과는 달리 난이도는 상당하다. 판 크기가 15*15 이하 정도로 작으면 보통은 숫자만 보고 푸는 것이 가능하지만, 난이도가 높아질수록 주어진 숫자만으로는 풀이가 힘들어지며 이런저런 논리적 방법을 사용해야 한다. 일반적으로 판 크기가 클수록 어려워지는 것은 당연하지만, 판 크기가 15*15 정도밖에 안 되는데도 굉장히 어려운 것도 있다.

숫자만큼 칸을 칠하는 것만큼이나 중요한 것은 칠해지지 않는 것이 확실한 칸을 확인해 두는 것이다. 노노그램이라는 이름도 이 특징에서 비롯된 것. 노노그램을 소재로 한 전자 게임에는 대체로 칠하지 않는 칸에 X표를 할 수 있는 기능이 존재한다.

숫자만 가지고 그려나갈 때, 가능한 경우가 2개 이상 나올 때가 있는데, 이럴 때는 한 가지 경우를 가정하고 맞는지 검사한다. 한편 논리적으로 풀이할 때도 어려운 문제의 경우 2가지 선택지 중 하나를 선택해야 할 때가 있는데, 이 때도 2개 중 한 쪽을 선택해 풀이해 나가며 모순이 발생하는지 확인하면 된다. 일종의 귀류법. 전자 게임에서는 이러한 상황에서 임시로 칠하는 칸을 확정된 칸과 별도로 표시할 수 있게 가정 모드가 존재하기도 한다.

침착맨 유튜브 편집자 1호부터 11호 정리

1호 침수자: 침투부 초기 ~ 2018

 영화산업의 꿈을 이루기 위해 하차. 클로즈업 편집과 55도발 히트다히트 등 무수히 많은 초기명작 남김.

​

2호 수석 침수자: 2018 ~ 현재

 공채 최종합격. 현재 수석 침수자로 활동 중이며 침수자 어셈블 권한 있음. 침투부 야외 컨텐츠 카메라 감독 및 사무실 매니저 및 기미상궁 겸임.

(서울과기대 디자인계열 졸업 예정)

​

3호 침수자: 2018 ~ 2019

 공채 아쉽게 최종 불합했으나 특별 채용. 굵직한 작품을 많이 남기고 침투부 아웃트로 제작하였으나, 대학 졸업위해 아쉽게 하차하신 양씨.

(건국대학교 서울캠 디자인계열)

​

4호 침수자: 2019 ~ 현재

 와우 클래식, 하스스톤등 게임 전문으로 편집 시작하였으나, 현재는 토크 등 올라운더로 가장 바쁘게 활약 중.

(서울대학교, 복수 전공)

​

5호 침수자: 2019 ~ 2019

 왕날편 사연 및 편집 전문으로 짧고 굵게 활약하였으나, 건강 상 이유로 하차함.

​

6호 침수자: 2019 ~ 현재

 5수자님을 이어 받아 왕날편 편집으로 커리어 시작, 현재 월드컵 등 올라운더 편집자로 활발히 활동하는 중.

​

7호 침수자: 2021 ~ 현재

 1호 침투부도우미에서 정식 7수자로 승격됨. 침착한조커, 쏘영이스페셜 등으로 유명한 어둠의 침수자 '얌숭'님

(건 당 작업)

​

8호 침수자: 2021 ~ 현재

 주로 '침착맨+' 서브 채널 게임 영상 및 쇼츠 편집. 각종 패러디 영상으로 유명한 어둠의 침수자 '슈말코'님

(건 당 작업)

​

9호 침수자: 2021 ~ 현재

 현재 침투부에서 가장 많은 활약하고 계신 신입 침수자님

(건 당 작업)

​

10호 침수자: 2021 ~ 현재

700명이 참여한 채용에서 당당히 합격. 장지수 김풍님 MBTI 영상 데뷔

(건 당 작업)

​

11호 침수자: 2021 ~ 현재

700명이 참여한 채용에서 당당히 합격. 고기동 얼음왕 주호민 영상으로 데뷔

(건 당 작업)


ㅊㅊ침까페


))))4호님이 빈지노 후배라 매우 친하다는 전설이..


김구영 구리시장 예비후보 “구리에는 관리형 리더가 아닌 기업가형 리더 필요”

김구영(61) 국민의힘 경기도당 수석대변인이 5일 구리시의회 멀티룸에서 기자회견을 열고 구리시장 선거 출마를 공식화했다.

지난달 27일 예비후보 등록을 마친 김 예비후보는 관리형 리더가 아닌 기업가형 리더의 필요성을 역설하며 구리만의 독점적 매력 창출을 강조했다.

30년 IT 기술 산업 현장 경험을 보유한 김 예비후보는 정체된 구리의 심장을 다시 뛰게 하겠다는 포부를 밝혔다. 스마트 문화도시로의 변모를 통해 이른바 ‘잃어버린 20년’을 회복하겠다는 구상이다. 주요 공약으로는 재정자립도 향상을 위한 경제 엔진 교체, 시민 체감형 복지 실현, 규제를 기회로 바꾸는 강점화 전략 등 3대 핵심 과제를 내놨다.

출마 배경에 대해서는 당초 당협위원장으로서 지방선거를 지휘하려 했으나, 중앙당의 위원장 교체 일정이 6월 이후로 연기됨에 따라 직접 출마를 결심했다고 설명했다. 아울러 구리시의 보수와 진보 비율을 52대 48로 분석하며 보수 결집 시 승리를 자신했다.

출처 : 인천일보(https://www.incheonilbo.com)

다만 잦은 당적 변경 이력은 당내 경선과 본선 과정에서 검증의 대상이 될 전망이다.

김 예비후보는 제8회 지방선거 당시 국민의힘 구리시장 경선에 참여한 뒤, 개혁신당 소속으로 제22대 총선에 출마했다가 지난해 국민의힘으로 복당한 바 있다. 이러한 이력이 당심과 표심에 미칠 영향이 이번 선거의 관건이다.

현장 기자회견에서는 IT 전문가로서의 구체적인 스마트 도시 실행 방안에 대한 질의가 이어졌다. 김 예비후보는 공공서류 무인 발급기 개발 참여 이력 등 실무 경험을 강조하며 현장성을 부각했다.

1964년 경기도 양평 출생인 김 예비후보는 경희대학교 공공대학원 행정학 석사를 취득했으며 현재 IT 기업을 경영하고 있다.

/구리=글·사진 박현기 기자 jcnews8090@incheonilbo.com

출처 : 인천일보(https://www.incheonilbo.com)

"""


In [80]:
pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 4.9 MB/s eta 0:00:00


In [92]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

text_splitter = SemanticChunker(OpenAIEmbeddings(),
                                breakpoint_threshold_type = "standard_deviation",
                                breakpoint_threshold_amount = 1.1)
chunks = text_splitter.split_text(texts)


In [94]:
print(len(chunks))
for i in range(len(chunks)):
  print(chunks[i])
  print("\n")
  print('-' * 50)
  print("\n")

7

1. 개요[편집]
노노그램예시
노노그램 예시

일본에서 개발된 퍼즐, 한국에서는 "네모로직", "네모네모로직"으로 불리기도 하는 퍼즐 게임. 영어로는 Nonograms(노노그램), Picross(피크로스), Griddlers 등으로 불린다. 평면만 있으면 할수 있는 게임으로 스도쿠와 함께 신문 같은 데서 찾아볼 수 있다. 규칙은 X×Y 크기[1]의 직사각형[2]에 각각 적혀 있는 숫자를 보고 숨어 있는 숫자를 예측해서 지우고 그려 나가면서 그림을 만들어 가는 게임. 2.


--------------------------------------------------


개발
1988년에 니시오 테츠야[3]와, 이시다 노부코[4]가 각각 창안하였다. 거의 비슷한 시기에 발표가 돼서 논쟁에 휩쓸리기도 했지만, 현재는 양쪽이 '같은 방식의 게임을 창작했다'로 인정한 상태. 한국에서는 제우미디어에서 출판을 전담하고 있다. 1권 ~ 6권까지는 니시오 테츠야의 퍼즐을 받아 직접 책으로 엮었으며, 7권부터는 오리지널 퍼즐도 수록하고 있다. 2021년 현재 기본편 43권(1~40, Plus[5] 1~3), 입문편 3권(1~2, 어린이용 1), 고급편 3권(1~2, Plus 1) 등 지속적으로 신간이 나오고 있는 중.


--------------------------------------------------


3.


--------------------------------------------------


방식[편집]
쓰인 숫자만큼의 연속된 칸을 칠해야 한다. 숫자와 숫자 사이에는 적어도 한칸을 비워야 한다. 숫자의 순서와 칠해진 칸의 순서는 일치해야 한다. 이렇게 간단한 규칙을 지니고 있다. 그러나 간단한 규칙과는 달리 난이도는 상당하다. 판 크기가 15*15 이하 정도로 작으면 보통은 숫자만 보고 푸는 것이 가능하지만, 난이도가 높아질수록 주어진 숫자만으로는 풀이가 힘들어지며 이런저런 논리적 방법을 사용해야 한다. 일반적으로 판 크기가 클수록 어려워지는 것은

In [88]:
chunks[1]

"개발\n1988년에 니시오 테츠야[3]와, 이시다 노부코[4]가 각각 창안하였다. 거의 비슷한 시기에 발표가 돼서 논쟁에 휩쓸리기도 했지만, 현재는 양쪽이 '같은 방식의 게임을 창작했다'로 인정한 상태. 한국에서는 제우미디어에서 출판을 전담하고 있다. 1권 ~ 6권까지는 니시오 테츠야의 퍼즐을 받아 직접 책으로 엮었으며, 7권부터는 오리지널 퍼즐도 수록하고 있다. 2021년 현재 기본편 43권(1~40, Plus[5] 1~3), 입문편 3권(1~2, 어린이용 1), 고급편 3권(1~2, Plus 1) 등 지속적으로 신간이 나오고 있는 중."

In [90]:
chunks[2]

'3.'